###Create Validation Views

This cell creates unified temporary views for Silver and Bronze data to compare and validate records.

In [0]:
-- Creates unified temporary views for Silver and Bronze data to compare and validate records.

CREATE OR REPLACE TEMP VIEW v_silver_all AS
SELECT 'blogs' AS tbl, content_id, url, topic, category, CAST(NULL AS STRING) AS difficulty_level, published_date, last_updated FROM edtech.silver.blogs
UNION ALL SELECT 'newsletters',      content_id, url, topic, category, NULL,             published_date, last_updated FROM edtech.silver.newsletters
UNION ALL SELECT 'coursera_courses', content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.coursera_courses
UNION ALL SELECT 'microsoft_learn',  content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.microsoft_learn
UNION ALL SELECT 'github_repos',     content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.github_repos
UNION ALL SELECT 'youtube_videos',   content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.youtube_videos;

-- Maps each Bronze row to its Silver table. Rows that match nothing get tbl = NULL.
CREATE OR REPLACE TEMP VIEW v_bronze_all AS
SELECT CASE LOWER(TRIM(content_type))
         WHEN 'article'    THEN 'blogs'
         WHEN 'newsletter' THEN 'newsletters' END AS tbl,
       content_id, title, url
FROM edtech.bronze.rss_raw
UNION ALL
SELECT CASE LOWER(TRIM(source))
         WHEN 'coursera'        THEN 'coursera_courses'
         WHEN 'microsoft learn' THEN 'microsoft_learn'
         WHEN 'github'          THEN 'github_repos'
         WHEN 'youtube'         THEN 'youtube_videos' END,
       content_id, title, url
FROM edtech.bronze.api_raw;

###Check Bronze vs Silver



In [0]:
-- Compares valid Bronze records with Silver records and identifies data gaps.

SELECT b.tbl, 
       b.bronze_rows, 
       b.bronze_valid_ids, 
       s.silver_rows, 
       b.bronze_valid_ids - s.silver_rows AS unexplained_gap 
FROM ( 
  -- Counts total and valid Bronze records
  SELECT tbl, 
         COUNT(*) AS bronze_rows, 
         COUNT(DISTINCT CASE WHEN NULLIF(TRIM(content_id),'') IS NOT NULL 
                              AND NULLIF(TRIM(title),'')      IS NOT NULL 
                              AND NULLIF(TRIM(url),'')        IS NOT NULL 
                             THEN TRIM(content_id) END) AS bronze_valid_ids 
  FROM v_bronze_all GROUP BY tbl 
) b 
LEFT JOIN (
  -- Counts records in Silver
  SELECT tbl, COUNT(*) AS silver_rows 
  FROM v_silver_all GROUP BY tbl
) s 
  ON b.tbl = s.tbl 
ORDER BY b.tbl;

tbl,bronze_rows,bronze_valid_ids,silver_rows,unexplained_gap
blogs,565,565,565,0
coursera_courses,300,300,300,0
github_repos,300,300,300,0
microsoft_learn,300,300,300,0
newsletters,35,35,35,0
youtube_videos,300,300,300,0


###Check Duplicate Content IDs



In [0]:
-- Uniqueness check: fails the cell immediately if any table has duplicate content_id.
SELECT assert_true(
  (SELECT MAX(dup) FROM (
     SELECT COUNT(*) - COUNT(DISTINCT content_id) AS dup
     FROM v_silver_all GROUP BY tbl
   )) = 0,
  'Duplicate content_id found in at least one Silver table'
) AS duplicate_content_ids_check;

duplicate_content_ids_check
null


### Check Duplicate URLs



In [0]:
-- Uniqueness check: fails if any table has duplicate url values.
SELECT assert_true(
  (SELECT MAX(dup) FROM (
     SELECT COUNT(*) - COUNT(DISTINCT url) AS dup
     FROM v_silver_all GROUP BY tbl
   )) = 0,
  'Duplicate url found in at least one Silver table'
) AS duplicate_urls_check;

duplicate_urls_check
null


### Check Bad URLs



In [0]:
-- Validity check: fails if any url does not start with http.
SELECT assert_true(
  (SELECT SUM(CASE WHEN url NOT LIKE 'http%' THEN 1 ELSE 0 END) FROM v_silver_all) = 0,
  'Malformed url found (does not start with http)'
) AS bad_urls_check;

bad_urls_check
null


### Check Date Range


In [0]:
-- Validity check: fails if any published_date is in the future or before year 2000.
SELECT assert_true(
  (SELECT SUM(CASE WHEN published_date > current_timestamp()
                      OR published_date < TIMESTAMP '2000-01-01'
                    THEN 1 ELSE 0 END)
   FROM v_silver_all) = 0,
  'published_date out of sane range found'
) AS out_of_range_dates_check;

out_of_range_dates_check
null


### Check Bronze to Silver Gap



In [0]:
-- Completeness check: fails if the count of valid Bronze rows doesn't match Silver rows.
SELECT assert_true(
  (SELECT MAX(ABS(gap)) FROM (
     SELECT b.tbl, b.bronze_valid_ids - COALESCE(s.silver_rows, 0) AS gap
     FROM (
       SELECT tbl,
              COUNT(DISTINCT CASE WHEN NULLIF(TRIM(content_id),'') IS NOT NULL
                                    AND NULLIF(TRIM(title),'')     IS NOT NULL
                                    AND NULLIF(TRIM(url),'')       IS NOT NULL
                                   THEN TRIM(content_id) END) AS bronze_valid_ids
       FROM v_bronze_all GROUP BY tbl
     ) b
     LEFT JOIN (
       SELECT tbl, COUNT(*) AS silver_rows FROM v_silver_all GROUP BY tbl
     ) s ON b.tbl = s.tbl
   )) = 0,
  'Unexplained gap between Bronze valid rows and Silver rows'
) AS bronze_to_silver_gap_check;

bronze_to_silver_gap_check
null


### Check for NaN Keywords


In [0]:
-- Validity check: fails if any 3-tier keyword extraction left the literal string "NaN".
SELECT assert_true(
  (SELECT SUM(nan_count) FROM (
     SELECT SUM(CASE WHEN list_of_keywords = 'NaN' THEN 1 ELSE 0 END) AS nan_count FROM edtech.silver.github_repos
     UNION ALL
     SELECT SUM(CASE WHEN list_of_keywords = 'NaN' THEN 1 ELSE 0 END) FROM edtech.silver.microsoft_learn
     UNION ALL
     SELECT SUM(CASE WHEN list_of_keywords = 'NaN' THEN 1 ELSE 0 END) FROM edtech.silver.youtube_videos
   )) = 0,
  'Leftover "NaN" string found in list_of_keywords'
) AS nan_keywords_check;

nan_keywords_check
null


### Check Missing Published Dates



In [0]:
-- Completeness check (warning only, not a hard gate): shows count of missing
-- published_date per table. Not wrapped in assert_true on purpose — a missing
-- date may be a real source-feed gap, not a pipeline bug, so we just observe it.
SELECT tbl, SUM(CASE WHEN published_date IS NULL THEN 1 ELSE 0 END) AS null_published
FROM v_silver_all
GROUP BY tbl;

tbl,null_published
blogs,0
newsletters,0
coursera_courses,0
microsoft_learn,0
github_repos,0
youtube_videos,0


##**Checks Gold data quality.**

In [0]:
-- Quality gate: fails if Gold has missing required fields or non-unique content_id.
SELECT assert_true(
  (SELECT COUNT(*) FROM edtech.gold.edtech_content
   WHERE content_id IS NULL OR TRIM(content_id) = ''
      OR title      IS NULL OR TRIM(title)      = ''
      OR source     IS NULL OR TRIM(source)     = ''
      OR url        IS NULL OR TRIM(url)        = ''
      OR published_date IS NULL) = 0
  AND
  (SELECT COUNT(*) - COUNT(DISTINCT content_id)
   FROM edtech.gold.edtech_content) = 0,
  'Gold has missing required fields or duplicate content_id'
) AS gold_completeness_check;

gold_completeness_check
null


**Checks URL validity.**

In [0]:
-- Quality gate: fails if any Gold URL is not a valid http(s) URL.
SELECT assert_true(
  (SELECT COUNT(*) FROM edtech.gold.edtech_content
   WHERE url IS NOT NULL
     AND NOT (url LIKE 'http://%' OR url LIKE 'https://%')) = 0,
  'Invalid URL found in Gold (not starting with http:// or https://)'
) AS gold_bad_urls_check;

gold_bad_urls_check
null


**Checks records by source.**

In [0]:
-- Checks the number of records from each source.

SELECT
    source,
    COUNT(*) AS record_count
FROM edtech.gold.edtech_content
GROUP BY source
ORDER BY record_count DESC;

source,record_count
Microsoft Learn,300
GitHub,300
Coursera,300
YouTube,300
Blog | OpenAI,237
Blog | Hugging Face,175
Blog | Kubernetes,47
Blog | Google Research,25
Newsletter | Data Science Weekly,18
Blog | AWS Compute,17


**Checks publication date.**

In [0]:
-- Checks Gold records by publication date.

SELECT
    content_id,
    title,
    source,
    published_date
FROM edtech.gold.edtech_content
ORDER BY published_date DESC NULLS LAST;

content_id,title,source,published_date
b1b8ff298bf1973e245de2aba58c23f6,From Megawatts to Tokens: How NVIDIA Maximizes AI Factory Production,Blog | NVIDIA,2026-09-15T16:55:59.000Z
2dc38c325368aedc062ceda312f7f0a1,"Scaling Federated Learning Across Docker, Kubernetes, and Slurm with NVIDIA FLARE",Blog | NVIDIA Developer,2026-09-15T15:00:00.000Z
QFlwFLmgR84,SQL + Excel Data Analyst Project | Complete Step-by-Step Tutorial #sql #excel #dataanalytics,YouTube,2026-09-15T10:30:20.000Z
3f68c3c1b921e8722c2f8a64f26cce28,Kubernetes v1.37: Memory QoS Graduates to Beta,Blog | Kubernetes,2026-09-14T18:30:00.000Z
125d85ffc03c1b55f5904c27c60622b4,Kubernetes Changed Block Tracking API - Beta Differences,Blog | Kubernetes,2026-09-14T18:30:00.000Z
vTVHasBQOdw,Complete Data Science Course FREE [2026] | Data Science Tutorial for Beginners | Intellipaat,YouTube,2026-09-14T17:02:20.000Z
103da91a3d020ead4f6350f13fe083f2,"AWS Weekly Roundup: OpenAI GPT-6 Astra on Amazon Bedrock, Amazon Quick desktop GA, Kiro for students, and more (September 14, 2026)",Blog | AWS News,2026-09-14T15:56:33.000Z
KrRDFpSjcR4,How to Write a Book with AI in 2026 (Full Step-By-Step Tutorial),YouTube,2026-09-14T12:00:35.000Z
4c3afbf85fca04de1372e13a7d2739e4,Article: Implementing Durable Workflows on Postgres Without an External Orchestrator,Blog | InfoQ Data Engineering,2026-09-14T11:00:00.000Z
sSkP75SHEfGPmxKUvFioHQ,Amazon Q Developer Beginner Training for Python,Coursera,2026-09-14T08:25:26.617Z


**Checks counts and duplicate.**

In [0]:
-- Checks row counts and duplicate records across all data layers.

SELECT
    'Bronze RSS' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_id) AS unique_ids,
    COUNT(*) - COUNT(DISTINCT content_id) AS duplicate_rows
FROM edtech.bronze.rss_raw

UNION ALL

SELECT
    'Bronze API',
    COUNT(*),
    COUNT(DISTINCT content_id),
    COUNT(*) - COUNT(DISTINCT content_id)
FROM edtech.bronze.api_raw

UNION ALL

SELECT
    'Silver Blogs',
    COUNT(*),
    COUNT(DISTINCT content_id),
    COUNT(*) - COUNT(DISTINCT content_id)
FROM edtech.silver.blogs

UNION ALL

SELECT
    'Silver Newsletters',
    COUNT(*),
    COUNT(DISTINCT content_id),
    COUNT(*) - COUNT(DISTINCT content_id)
FROM edtech.silver.newsletters

UNION ALL

SELECT
    'Silver Coursera',
    COUNT(*),
    COUNT(DISTINCT content_id),
    COUNT(*) - COUNT(DISTINCT content_id)
FROM edtech.silver.coursera_courses

UNION ALL

SELECT
    'Silver Microsoft Learn',
    COUNT(*),
    COUNT(DISTINCT content_id),
    COUNT(*) - COUNT(DISTINCT content_id)
FROM edtech.silver.microsoft_learn

UNION ALL

SELECT
    'Silver GitHub',
    COUNT(*),
    COUNT(DISTINCT content_id),
    COUNT(*) - COUNT(DISTINCT content_id)
FROM edtech.silver.github_repos

UNION ALL

SELECT
    'Silver YouTube',
    COUNT(*),
    COUNT(DISTINCT content_id),
    COUNT(*) - COUNT(DISTINCT content_id)
FROM edtech.silver.youtube_videos

UNION ALL

SELECT
    'Gold',
    COUNT(*),
    COUNT(DISTINCT content_id),
    COUNT(*) - COUNT(DISTINCT content_id)
FROM edtech.gold.edtech_content;

table_name,total_rows,unique_ids,duplicate_rows
Bronze RSS,600,600,0
Bronze API,1200,1200,0
Silver Blogs,565,565,0
Silver Newsletters,35,35,0
Silver Coursera,300,300,0
Silver Microsoft Learn,300,300,0
Silver GitHub,300,300,0
Silver YouTube,300,300,0
Gold,1800,1800,0


In [0]:
%python
import json
from datetime import datetime

# Reaching this cell means every assert_true check above passed —
# both the 6 Silver checks and the 2 new Gold gates. If any had failed,
# the notebook would have stopped before this point.

quality_report = {
    "status": "PASS",
    "checks": [
        # --- Silver gates ---
        "duplicate_content_ids",
        "duplicate_urls",
        "bad_urls",
        "out_of_range_dates",
        "bronze_to_silver_gap",
        "nan_keywords",
        # --- Gold gates ---
        "gold_completeness",
        "gold_bad_urls",
    ],
    "layers": {
        "silver": 6,
        "gold": 2,
    },
    "last_run": datetime.now().isoformat(),
}

with open("/Volumes/edtech/gold/exports/quality_report.json", "w") as f:
    json.dump(quality_report, f, indent=2)

print("Saved:", quality_report)

Saved: {'status': 'PASS', 'checks': ['duplicate_content_ids', 'duplicate_urls', 'bad_urls', 'out_of_range_dates', 'bronze_to_silver_gap', 'nan_keywords', 'gold_completeness', 'gold_bad_urls'], 'layers': {'silver': 6, 'gold': 2}, 'last_run': '2026-09-26T21:22:33.735847'}
